# Phase 4: Facility Manager Agent (Subscriber + Publisher)

This notebook subscribes to spectator events and publishes queue-state updates for Section A4.

In [1]:
# Cell 2 purpose: Import modules and ensure src is available from notebooks/.
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path('../src').resolve()))

from simulated_city import mqtt
from simulated_city.config import load_config
from simulated_city.facility_manager import FacilityManagerState, process_spectator_event
from simulated_city.topic_schema import topic_queue_state, topic_spectator_events

In [5]:
# Cell 3 purpose: Load config, create manager state, and connect MQTT client.
app_config = load_config()
if app_config.halftime is None:
    raise ValueError('Missing halftime section in config.yaml')

state = FacilityManagerState(shared_urinal_total=app_config.halftime.capacity.shared_urinal_total)

spectator_topic = topic_spectator_events()
queue_state_topic = topic_queue_state()
mqtt_client = mqtt.connect_mqtt(app_config.mqtt, client_id_suffix='facility-manager')

print(f'Connected to MQTT broker at {app_config.mqtt.host}:{app_config.mqtt.port}')
print(f'Subscribed topic: {spectator_topic}')
print(f'Publish topic: {queue_state_topic}')

Connected to MQTT broker at 127.0.0.1:1883
Subscribed topic: stadium/a4/halftime/events/spectator
Publish topic: stadium/a4/halftime/state/queues


In [6]:
# Cell 4 purpose: Subscribe to spectator events and process incoming messages into queue-state payloads.
received_events = []
published_states = []

def _on_message(client, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode('utf-8'))
    except json.JSONDecodeError:
        return

    received_events.append(payload)
    queue_state_payload = process_spectator_event(state, payload)
    if queue_state_payload is None:
        return

    ok = mqtt.publish_json_checked(client, queue_state_topic, queue_state_payload, qos=1)
    if ok:
        published_states.append(queue_state_payload)

mqtt_client.on_message = _on_message
mqtt_client.subscribe(spectator_topic, qos=1)
print('Subscription started. Waiting for incoming spectator events...')

Subscription started. Waiting for incoming spectator events...


In [4]:
# Cell 5 purpose: Run a short processing loop, then print and disconnect safely.
run_for_s = 30
start = time.time()
while time.time() - start < run_for_s:
    time.sleep(0.5)

print(f'Received spectator events: {len(received_events)}')
print(f'Published queue states: {len(published_states)}')
if published_states:
    last = published_states[-1]
    print('Last published queue-state payload:')
    print(last)
    zone_a = last['queues']['zone_a']
    zone_b = last['queues']['zone_b']
    print('Zone usage snapshot (last payload):')
    print(f"  Zone 1 -> toilet={zone_a['toilet']}, cafe={zone_a['cafe']}")
    print(f"  Zone 2 -> toilet={zone_b['toilet']}, cafe={zone_b['cafe']}")
    print(f"  Shared urinal -> {last['queues']['shared_mens_urinal']}")

connector = getattr(mqtt_client, '_simcity_connector', None)
if connector is not None:
    connector.disconnect()
    print('Disconnected from MQTT broker.')

Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


Received spectator events: 181
Published queue states: 181
Last published queue-state payload:
{'schema_version': '1.0', 'run_id': 'a4-run-eb25aa71', 'timestamp_s': 900, 'source_event_timestamp_s': 900, 'queues': {'zone_a': {'toilet': 85, 'cafe': 65}, 'zone_b': {'toilet': 71, 'cafe': 57}, 'shared_mens_urinal': 48}}
Zone usage snapshot (last payload):
  Zone 1 -> toilet=85, cafe=65
  Zone 2 -> toilet=71, cafe=57
  Shared urinal -> 48
Disconnected from MQTT broker.
